# Solution C3 · The optimism of best_score_: measuring it with a simulation

**Type**: Break the model

## Objective

Measure, on 16 simulated datasets with a weak signal, by how much the `best_score_` of a `GridSearchCV` exceeds the true performance of the
chosen model, and verify that a nested cross-validation estimates it without bias.

In [ ]:
import sys, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "common").is_dir())
sys.path.insert(0, str(_root))

import warnings; warnings.filterwarnings("ignore")     # clean output: here the warnings are just noise
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from common import datasets as ds
from common.verify import check, close, done, verify

from sklearn.datasets import make_classification
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import GridSearchCV, StratifiedKFold, cross_val_score
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

estimator = make_pipeline(StandardScaler(), KNeighborsClassifier())
grid = {"kneighborsclassifier__n_neighbors": [1, 5, 11, 17, 25, 35],
        "kneighborsclassifier__weights": ["uniform", "distance"],
        "kneighborsclassifier__p": [1, 2]}                       # 6 x 2 x 2 = 24 combinations


def generate(seed):
    X, y = make_classification(n_samples=2120, n_features=15, n_informative=2, n_redundant=0, n_clusters_per_class=1,
                               class_sep=0.35, flip_y=0.2, random_state=seed)
    return X[:120], y[:120], X[120:], y[120:]                       # development (120) and truth (2000)

## Commented solution

In [ ]:
def experiment(seed):
    X_dev, y_dev, X_true, y_true = generate(seed)
    cv = StratifiedKFold(5, shuffle=True, random_state=seed)
    search = GridSearchCV(estimator, grid, cv=cv, scoring="roc_auc").fit(X_dev, y_dev)
    best_score = search.best_score_                                                      # the estimate "of the search"
    true_auc = roc_auc_score(y_true, search.predict_proba(X_true)[:, 1])                 # the performance on never-seen data
    inner = GridSearchCV(estimator, grid, cv=3, scoring="roc_auc")                       # the search becomes part of the model
    nested_estimate = cross_val_score(inner, X_dev, y_dev, cv=cv, scoring="roc_auc").mean()
    return best_score, true_auc, nested_estimate

In [ ]:
results = pd.DataFrame([experiment(s) for s in range(16)], columns=["best_score", "true_auc", "nested"])
best_optimism = (results["best_score"] - results["true_auc"]).mean()
nested_error = (results["nested"] - results["true_auc"]).mean()
print(results.mean().round(3).to_dict())
print(f"optimism of best_score_: {best_optimism:+.3f} | error of the nested estimate: {nested_error:+.3f}")

## Check

In [ ]:
with verify():
    check(results.shape == (16, 3), "16 experiments", "one row per seed")
    check((results["best_score"] > results["true_auc"]).mean() > 0.8, "best_score exceeds the true performance in the great majority of the datasets",
          "in over 80% of the datasets best_score_ should exceed the AUC on the truth dataset")
    check(close(best_optimism, (results.best_score - results.true_auc).mean()) and best_optimism > 0.04,
          "the mean optimism of best_score_ exceeds 0.04 of AUC", "optimism = mean(best_score - true_auc)")
    check(close(nested_error, (results.nested - results.true_auc).mean()) and abs(nested_error) < best_optimism / 2,
          "the nested estimate is much less biased than best_score_ (less than half of the optimism)", "error = mean(nested - true_auc)")
done()

## What you learned

- With only 24 candidates and 5 folds, `best_score_` overestimates the true performance by about 0.04 of AUC (from 0.681 to 0.638 on average) in almost
  every dataset. It is the maximum of 24 noisy estimates: choosing the winner also selects its luck.
- The optimism grows with the number of combinations tried and with the smallness of the dataset: here 120 rows and a weak signal are
  the worst case, but the mechanism is always there.
- **Nested cross-validation** (the search inside every outer fold) estimates the performance of the *whole procedure* "search and train"
  almost without bias: here 0.628 against 0.638 (an error of −0.01, slightly pessimistic because the inner search uses less
  data). It costs outer_folds × trainings of the search: here 5 × 72 = 360 trainings instead of 120.

## Common pitfall

Reporting `best_score_` in a report, or choosing between two tuning procedures by looking at their `best_score_`: you compare scores
inflated in different ways. To compare procedures you need the nested estimate (or an untouched test set).

## Interview question

> *Click on a question to show its answer.*

<details><summary>When do you really need a nested cross-validation, and when is a test set enough?</summary>

It is needed when the data are few and I cannot afford a separate test set of sufficient size, or when I have to compare different
tuning procedures and report an unbiased estimate (for example in a publication or in a regulatory review). With plenty of data a test set held out and touched once is enough: it costs much less and does not require
repeating the search in every fold.

</details>